# 01 - Download BCB SGS data

Download all manually verified Banco Central do Brasil SGS series listed in `data/series_dictionary.csv`.

In [3]:
import sys
print(sys.executable)

import pandas as pd
print(pd.__version__)

c:\Users\chico\brazil-directed-credit-monetary-policy\.venv-1\Scripts\python.exe
2.3.3


In [4]:
from datetime import date
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bcb_api import fetch_sgs_series, save_series_csv

In [5]:
START_DATE = "2011-01-01"
END_DATE = date.today().isoformat()

SERIES_DICTIONARY_PATH = PROJECT_ROOT / "data" / "series_dictionary.csv"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
COMBINED_OUTPUT_PATH = RAW_DATA_DIR / "bcb_sgs_all_long.csv"

In [6]:
series_dictionary = pd.read_csv(SERIES_DICTIONARY_PATH)
verified_series = series_dictionary.loc[
    series_dictionary["verified"].astype(str).str.lower().eq("yes")
].copy()

verified_series[["name", "series_id", "frequency", "unit"]]

,name,series_id,frequency,unit
0,selic_target,432,monthly,percent_annual
1,central_bank_base_rate_tbc,422,monthly,percent_annual
2,central_bank_assistance_rate_tban,423,monthly,percent_annual
3,inflation_target,13521,monthly_or_annual,percent_annual
4,ipca,433,monthly,percent_monthly
5,ipca_12m,13522,monthly,percent_12_months
6,industrial_output_general,21859,monthly,index_2022_100
7,gdp_monthly_current_prices,4380,monthly,current_brl
8,exchange_rate_commercial_buy_usd,10813,daily,cmu_per_usd
9,credit_total_stock,20539,monthly,brl


In [7]:
downloaded = []
summary_rows = []

for row in verified_series.itertuples(index=False):
    df = fetch_sgs_series(
        series_id=row.series_id,
        start_date=START_DATE,
        end_date=END_DATE,
        name=row.name,
    )

    output_path = RAW_DATA_DIR / f"{row.name}.csv"
    save_series_csv(df, output_path)
    downloaded.append(df)

    summary_rows.append(
        {
            "series": row.name,
            "series_id": row.series_id,
            "observations": len(df),
            "first_date": df["date"].min() if not df.empty else pd.NaT,
            "last_date": df["date"].max() if not df.empty else pd.NaT,
        }
    )

if downloaded:
    combined = pd.concat(downloaded, ignore_index=True)
else:
    combined = pd.DataFrame(columns=["date", "value", "series"])

save_series_csv(combined, COMBINED_OUTPUT_PATH)

summary = pd.DataFrame(summary_rows)
summary["first_date"] = pd.to_datetime(summary["first_date"]).dt.date
summary["last_date"] = pd.to_datetime(summary["last_date"]).dt.date

print(summary.to_string(index=False))

                                              series  series_id  observations first_date  last_date
                                        selic_target        432             0        NaT        NaT
                          central_bank_base_rate_tbc        422             0        NaT        NaT
                   central_bank_assistance_rate_tban        423             0        NaT        NaT
                                    inflation_target      13521            16 2011-01-01 2026-01-01
                                                ipca        433           184 2011-01-01 2026-04-01
                                            ipca_12m      13522           184 2011-01-01 2026-04-01
                           industrial_output_general      21859           183 2011-01-01 2026-03-01
                          gdp_monthly_current_prices       4380           184 2011-01-01 2026-04-01
                    exchange_rate_commercial_buy_usd      10813             0        NaT        NaT


C:\Users\chico\AppData\Local\Temp\ipykernel_28312\3067282182.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(downloaded, ignore_index=True)
